# Company Web Scraping with Schematron

This example shows how to extract data from a website using Schematron. In this example, we'll be extracting product and company info from Ramp's homepage.

Schematron is a model trained specifically to convert messy HTML into clean, structured JSON. You just define a schema, and it extracts the data - no prompts needed.

**Documentation:** [JSON Extraction Guide](https://docs.inference.net/use-cases/json-extraction#json-extraction)

**Announcement Blog:** [Introducing Schematron](https://inference.net/blog/Schematron)

**Model Pages:**
- [Schematron-8B](https://inference.net/models/schematron-8b)
- [Schematron-3B](https://inference.net/models/schematron-3b)

## Let's get started

### Install Dependencies

We need:
- `openai` - To call the Inference.net API
- `pydantic` - To define our data schema
- `lxml_html_clean` - To clean up HTML before processing
- `requests` - To fetch websites


In [ ]:
%pip install openai pydantic lxml_html_clean requests

### Setup the API Client

Configure the OpenAI client to point to Inference.net. Get your Inference.net API key at: https://inference.net/dashboard/api-keys

In [45]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://api.inference.net/v1",
    api_key=os.getenv("INFERENCE_API_KEY"),
)

### Define Your Schema

Use Pydantic to define what information you want to extract. The field descriptions help guide the extraction.


In [ ]:
from pydantic import BaseModel, Field

class Product(BaseModel):
    name: str = Field(..., description="Product name")
    description: str = Field(..., description="What the product does")

class CompanyInfo(BaseModel):
    company_name: str = Field(..., description="Company name")
    tagline: str = Field(..., description="Main headline or tagline")
    description: str = Field(..., description="What the company does")
    products: list[Product] = Field(default_factory=list, description="List of products")
    target_customers: list[str] = Field(default_factory=list, description="Customer segments like 'Startups', 'Enterprise'")
    customer_count: str = Field(default="", description="Number of customers if mentioned")


### Fetch the Website

Let's grab the HTML from Ramp's homepage.


In [19]:
import requests

url = "https://ramp.com"
response = requests.get(url, headers={
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
})
html = response.text

print(f"Fetched {len(html):,} characters from {url}")

Fetched 2,097,646 characters from https://ramp.com


### Clean the HTML (Recommended)

Schematron was trained on HTML that had scripts and styles removed. Cleaning the HTML improves extraction quality.

In this example, the raw html is way outside the context window of Schematron, which 128k tokens. Most of the html is junk like crazy css classes/modules, link/script tags, svg-codes, and data- and aria- attributes. We can safely remove all of this.

There are multiple libraries that can do this cleaning, but Schematron was trained on websites cleaned with lxml, so we'll use that here. The most important thing is that the information we are trying to extract isn't filtered out.

In [24]:
from lxml_html_clean import Cleaner
import lxml.html as LH

cleaner = Cleaner(
    scripts=True,
    javascript=True,
    style=True,
    inline_style=True,
)

doc = LH.fromstring(html)
cleaned_doc = cleaner.clean_html(doc)
cleaned_html = LH.tostring(cleaned_doc, encoding="unicode")

print(f"Cleaned HTML: {len(cleaned_html):,} characters")


Cleaned HTML: 102,204 characters


### Extract with Schematron

Now we call Schematron with our schema. Note:
- No prompt needed - the schema guides extraction
- Temperature should be 0 for best results
- Use `schematron-8b` for best quality, or `schematron-3b` for lower cost


In [46]:
response = client.beta.chat.completions.parse(
    model="inference-net/schematron-8b",
    messages=[
        {"role": "user", "content": cleaned_html},
    ],
    response_format=CompanyInfo,
    temperature=0,
)

company = response.choices[0].message.parsed

### View the Results

Let's see what Schematron extracted from Ramp's website!


In [47]:
import json

print(json.dumps(company.model_dump(), indent=2))

{
  "company_name": "Ramp",
  "tagline": "Time is money. Save both.",
  "description": "Ramp is a platform designed to make finance teams faster and happier by consolidating spend management, corporate cards, bill payments, accounting, and more into a single, easy-to-use system. The platform automates workflows, integrates with accounting systems, and provides global payment capabilities, helping businesses of all sizes save time and money.",
  "products": [
    {
      "name": "Ramp Intelligence",
      "description": "AI-powered tools to automate spend management and detect out-of-policy transactions."
    },
    {
      "name": "Corporate Cards",
      "description": "Control spend before it happens with customizable cards and built-in controls."
    },
    {
      "name": "Expense Management",
      "description": "Automated expense submission and approval workflows to streamline expense reporting."
    },
    {
      "name": "Travel",
      "description": "Travel management that e

### Pretty Print

It's pretty hard to read the above json, so let's format it a bit better for easy reading.

In [48]:
print(f"Company: {company.company_name}")
print(f"Tagline: {company.tagline}")
print(f"\nDescription:\n{company.description}")

print(f"\nProducts ({len(company.products)}):")
for p in company.products:
    print(f"  • {p.name}: {p.description}")

print(f"\nTarget Customers: {', '.join(company.target_customers)}")
print(f"Customer Count: {company.customer_count}")


Company: Ramp
Tagline: Time is money. Save both.

Description:
Ramp is a platform designed to make finance teams faster and happier by consolidating spend management, corporate cards, bill payments, accounting, and more into a single, easy-to-use system. The platform automates workflows, integrates with accounting systems, and provides global payment capabilities, helping businesses of all sizes save time and money.

Products (15):
  • Ramp Intelligence: AI-powered tools to automate spend management and detect out-of-policy transactions.
  • Corporate Cards: Control spend before it happens with customizable cards and built-in controls.
  • Expense Management: Automated expense submission and approval workflows to streamline expense reporting.
  • Travel: Travel management that ensures all expenses are in policy.
  • Accounts Payable: Process bills in seconds with automated AP workflows.
  • Procurement: Run intake-to-pay without delay with streamlined procurement processes.
  • Account

## That's It!

You've successfully extracted structured company data from a website using Schematron.

### Key Takeaways

- **No prompts needed** - Just define your schema
- **100% valid JSON** - Always matches your schema
- **Long context** - Handles up to 128K tokens
- **Cost-efficient** - Much cheaper than using GPT-5 for extraction

### Try It Yourself

Change the `url` variable to scrape other companies, or modify the `CompanyInfo` schema to extract different fields!
